In [ ]:
import pandas as pd
import numpy as np
import miceforest as mf
from datetime import datetime
from lifelines.statistics import logrank_test
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

In [ ]:
df = pd.read_csv("CLEAN_DEEP.csv")

In [ ]:
df = df.drop(columns="MEASUREMENT_DATE_TRR")

In [ ]:
df_for_codes = df.copy()

In [ ]:
bin_cols = [col for col in df.columns if df[col].nunique() == 2]
print(bin_cols)

In [ ]:
#make sure to remove GSTATUS from the bin_cols group
#removing CTR_CODE and TX_DATE as they will be dropped soon

cat_cols = ['GENDER', 'SUD_DEATH', 'IMPL_DEFIBRIL', 'INFECT_IV_DRUG_TRR', 'INOTROPES_TRR', 'OTH_LIFE_SUP_TRR', 
             'STEROID', 'TRANSFUSIONS', 'VENT_SUPPORT_TRR', 'VENTILATOR_TRR', 'HBV_SURF_TOTAL', 'CMV_STATUS', 
             'EBV_SEROSTATUS', 'GENDER_DON', 'ANTIHYPE_DON', 'BLOOD_INF_DON', 'OTHER_INF_DON', 
             'PT_DIURETICS_DON', 'PT_STEROIDS_DON', 'PT_T4_DON', 'PULM_INF_DON', 'URINE_INF_DON', 'VASODIL_DON', 
             'CLIN_INFECT_DON', 'HIST_OTH_DRUG_DON', 'CMV_DON', 'DDAVP_DON', 'ARGININE_DON', 'INSULIN_DON', 
             'LIFE_SUP_TRR', 'PRIOR_TH_SURG_TRR', 'PROTEIN_URINE', 'CARDARREST_NEURO', 'EBV_IGG_CAD_DON', 
             'CDC_RISK_HIV_DON', 'INOTROP_SUPPORT_DON'] + ["END_STAT", 
                "ETHCAT", "ETHCAT_DON", "ACADEMIC_LEVEL_TRR", "ACADEMIC_PRG_TRR", 
            "FUNC_STAT_TRR", "MED_COND_TRR", "PRI_PAYMENT_TRR", "COGNITIVE_DEV_TRR", "MOTOR_DEV_TRR", "COD_CAD_DON", 
            "ABO_MAT", "DIAG", "PROC_TY_HR", "TRANSFUS_TERM_DON", "VAD_DEVICE_TY_TRR"
            #  , "CTR_CODE"
             ]

outcome = ["GSTATUS", "GTIME"]

# Create a single list of all columns to exclude
exclude = cat_cols + outcome

# Filter df.columns for those not in the exclude list
print([col for col in df.columns if col not in exclude])

In [ ]:
num_cols = ['TOT_SERUM_ALBUM', 'DAYS_STAT1A', 'DAYS_STAT2', 'DAYS_STAT1B', 'CREAT_TRR', 'HEMO_PA_MN_TRR', 
            'TBILI', 'CPRA', 'CPRA_PEAK', 'HLAMIS', 'AGE_DON', 'BUN_DON', 'CREAT_DON', 'SGOT_DON', 'SGPT_DON', 
            'TBILI_DON', 'HGT_CM_DON_CALC', 'WGT_KG_DON_CALC', 
            # 'TX_DATE', 
            'AGE', 'ISCHTIME', 'HGT_CM_CALC', 
            'WGT_KG_CALC', 'PO2', 'LV_EJECT', 'PO2_FIO2_DON', 'PCO2_DON', 'PH_DON', 'HEMATOCRIT_DON']


In [ ]:
for col in cat_cols:
    # ensure it’s a pandas Categorical
    df[col] = df[col].astype('category')
    # get integer codes (-1 == NaN)
    codes = df[col].cat.codes
    # replace -1 with real NaN and cast to float so MissForest sees the missing
    df[col] = codes.replace(-1, np.nan).astype(float)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.25, random_state=42)

train_df = train_df.drop(columns="TX_DATE")

test_df = test_df.drop(columns="TX_DATE")

In [ ]:
print(train_df.shape)
print(test_df.shape)

In [ ]:
# Calculate frequencies on training data
train_counts = train_df['CTR_CODE'].value_counts()

# Convert the resulting Series into a dictionary
count_dict = train_counts.to_dict()

# Get quartile ranks (0, 1, 2, 3) for each center in the training counts
# We use the unique index of our count_dict (the ctr_codes)
quartile_ranks = pd.qcut(train_counts, q=4, labels=False, duplicates='drop')

# Create your final lookup dictionary: {ctr_code: quartile_rank}
quartile_map_dict = quartile_ranks.to_dict()

# Create the new column in both DataFrames
train_df['ctr_quartile'] = train_df['CTR_CODE'].map(quartile_map_dict).astype(int)
test_df['ctr_quartile'] = test_df['CTR_CODE'].map(quartile_map_dict)

# Handle potential missing centers in the test set (centers not seen in train)
# Using -1 or 0 to represent "Unknown/New Center"
test_df['ctr_quartile'] = test_df['ctr_quartile'].fillna(1).astype(int)



train_df = train_df.drop(columns="CTR_CODE")
test_df = test_df.drop(columns="CTR_CODE")

train_df[cat_cols] = train_df[cat_cols].astype('category')
test_df[cat_cols] = test_df[cat_cols].astype('category')

train_df["ctr_quartile"] = train_df["ctr_quartile"].astype('category')
test_df["ctr_quartile"] = test_df["ctr_quartile"].astype('category')



In [ ]:
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

y_train = train_df[outcome]
x_train = train_df.drop(columns=outcome, axis=1)

y_test = test_df[outcome]
x_test = test_df.drop(columns=outcome, axis=1)

In [ ]:
x_test["PROC_TY_HR"] = (
    x_test["PROC_TY_HR"]
    .astype("category")
    .cat.set_categories(x_train["PROC_TY_HR"].cat.categories)
)
x_test["ctr_quartile"] = x_test["ctr_quartile"].astype(x_train["ctr_quartile"].dtype)



In [ ]:
kernel = mf.ImputationKernel(
    x_train,
    random_state=1,
    mean_match_candidates=0  # <-- disables KDTree mean matching
)
kernel.mice(5)

# Impute TRAIN
train_imputed = kernel.complete_data(dataset=0)


In [ ]:
# Impute TEST using the trained kernel
# miceforest provides "impute_new_data" for this purpose
test_kernel = kernel.impute_new_data(x_test)
test_imputed = test_kernel.complete_data(dataset=0)

In [ ]:
x_att = train_imputed.copy()
x_att_test = test_imputed.copy()

# cat_cols_2 = ['GENDER', 'IMPL_DEFIBRIL', 'INFECT_IV_DRUG_TRR', 'INOTROPES_TRR', 'OTH_LIFE_SUP_TRR', 'STEROID', 
#              'TRANSFUSIONS', 'VENT_SUPPORT_TRR', 'VENTILATOR_TRR', 'CMV_STATUS', 'EBV_SEROSTATUS', 'GENDER_DON', 'ANTIHYPE_DON', 'BLOOD_INF_DON', 'OTHER_INF_DON', 'PT_DIURETICS_DON', 'PT_STEROIDS_DON', 'PT_T4_DON', 'PULM_INF_DON', 'URINE_INF_DON', 'VASODIL_DON', 'CLIN_INFECT_DON', 'HIST_OTH_DRUG_DON', 'CMV_DON', 'DDAVP_DON', 'ARGININE_DON', 'INSULIN_DON', 'LIFE_SUP_TRR', 'PRIOR_TH_SURG_TRR', 'PROTEIN_URINE', 'CARDARREST_NEURO', 'EBV_IGG_CAD_DON', 'CDC_RISK_HIV_DON', 'INOTROP_SUPPORT_DON'] + ["END_STAT", "ETHCAT", "ETHCAT_DON", "ACADEMIC_LEVEL_TRR", "ACADEMIC_PRG_TRR", 
#             "FUNC_STAT_TRR", "MED_COND_TRR", "PRI_PAYMENT_TRR", "COGNITIVE_DEV_TRR", "MOTOR_DEV_TRR", "COD_CAD_DON", 
#             "ABO_MAT", "DIAG", "PROC_TY_HR", "TRANSFUS_TERM_DON", "VAD_DEVICE_TY_TRR"
#             ]
for col in cat_cols:
    # 1. grab the original category labels
    cats = df_for_codes[col].astype('category').cat.categories
    # 2. pull out your integer codes
    codes = x_att[col].round().astype(int)
    # 3. rebuild a Categorical from those codes + original labels
    x_att[col] = pd.Categorical.from_codes(codes, categories=cats)

In [ ]:
imputed_train = pd.concat([x_att, y_train], axis=1)
imputed_test = pd.concat([x_att_test, y_test], axis=1)
#due to the nature of the random split, variables with very low missingness may have no missing values in the train set
#   but some in the test set, MICE can't deal with this and this is why we resort to mean and mode imp for these values

def fill_test_missing_from_train(x_train: pd.DataFrame, x_test: pd.DataFrame):
    x_test = x_test.copy()

    # columns that have missing values in TEST
    cols_with_missing = x_test.columns[x_test.isna().any()]

    for col in cols_with_missing:
        # skip if TRAIN also has missingness (MICE should have handled those)
        if x_train[col].isna().any():
            continue

        # numeric → median
        if pd.api.types.is_numeric_dtype(x_train[col]):
            fill_value = x_train[col].median()

        # categorical / object → mode
        else:
            fill_value = x_train[col].mode().iloc[0]

        x_test[col] = x_test[col].fillna(fill_value)

    return x_test

imputed_test = fill_test_missing_from_train(imputed_train, imputed_test)

#add weight ratios

imputed_train["weight_ratio"] = imputed_train["WGT_KG_DON_CALC"] / imputed_train["WGT_KG_CALC"]
imputed_test["weight_ratio"] = imputed_test["WGT_KG_DON_CALC"] / imputed_test["WGT_KG_CALC"]

In [ ]:
imputed_train.to_csv("imputed_train.csv", index=False)
imputed_test.to_csv("imputed_test.csv", index=False)